# Data leakage
Leakage lets future or target-derived information enter training, making validation look falsely good.

Data leakage asks the empirical-risk machinery to respect time, provenance, and the validation boundary. The training number can look excellent for the wrong reason when target-derived or future information slips into the features.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import log_loss
from sklearn.metrics import recall_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
np.random.seed(7)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def safe_split(X, y, test_size=0.4):
    counts = np.bincount(np.asarray(y))
    can_stratify = counts.min() >= 2
    stratify = y if can_stratify else None
    return train_test_split(X, y, test_size=test_size, random_state=0, stratify=stratify)


def scaled_train_test(X, y, test_size=0.4):
    x_tr, x_te, y_tr, y_te = safe_split(X, y, test_size=test_size)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def logistic_model(**kwargs):
    params = dict(max_iter=3000, solver="lbfgs")
    params.update(kwargs)
    return LogisticRegression(**params)


def predict_proba_or_scores(model, X, labels):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
    else:
        scores = model.decision_function(X)
        if scores.ndim == 1:
            scores = np.column_stack([-scores, scores])
        scores = scores - scores.max(axis=1, keepdims=True)
        proba = np.exp(scores)
        proba = proba / proba.sum(axis=1, keepdims=True)
    if proba.shape[1] == len(labels):
        return proba
    aligned = np.zeros((len(X), len(labels)))
    for j, label in enumerate(model.classes_):
        idx = list(labels).index(label)
        aligned[:, idx] = proba[:, j]
    aligned = np.clip(aligned, 1e-9, 1.0)
    aligned = aligned / aligned.sum(axis=1, keepdims=True)
    return aligned


def model_log_loss(model, x_te, y_te, labels):
    proba = predict_proba_or_scores(model, x_te, labels)
    return float(log_loss(y_te, proba, labels=labels))


def print_table(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    fmt = "  ".join("{:" + str(w) + "}" for w in widths)
    print(fmt.format(*headers))
    print(fmt.format(*["-" * w for w in widths]))
    for row in rows:
        print(fmt.format(*row))


def plot_summary(names, metrics, title, ylabel):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    flat = axes.ravel()
    for idx, (name, X, y) in enumerate(clf_ladder()):
        ax = flat[idx]
        sample = X[:, :2]
        ax.scatter(sample[:, 0], sample[:, 1], c=y, cmap="viridis", s=20, alpha=0.8)
        ax.set_title(name.split("(")[0].strip())
        ax.set_xticks([])
        ax.set_yticks([])
    ax = flat[-1]
    ax.plot(range(1, len(metrics) + 1), metrics, marker="o")
    ax.set_title(title)
    ax.set_xlabel("rung")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, len(metrics) + 1))
    fig.tight_layout()
    plt.show()


## The concept, built once on D1
The lesson formula is $$R_{leaky}=\frac1m\sum_i \ell(f(x_i,\text{future}_i),y_i)$$
We first reproduce the lesson arithmetic exactly, then reuse the corresponding real technique below.

In [ ]:

def data_leakage_method(losses, cost, alternative):
    risk = float(np.mean(losses))
    score = risk + cost
    gap = alternative - score
    return risk, score, gap

losses = np.array([0.257, 0.122, 0.420])
risk, score, gap = data_leakage_method(losses, 0.060, 0.374)
print(f"lesson risk={risk:.3f}, score={score:.3f}, gap={gap:.3f}")
assert round(risk, 3) == 0.266
assert round(score, 3) == 0.326
assert round(gap, 3) == 0.048


Now connect the arithmetic to a reusable model-selection habit: compute the validation metric, add any stated cost, and compare the gap to an alternative. The assert guards make the notebook reproducible.

In [ ]:
lesson_losses = np.array([0.257, 0.122, 0.42])
lesson_risk, lesson_score, lesson_gap = data_leakage_method(lesson_losses, 0.060, 0.374)
print(round(lesson_risk, 3), round(lesson_score, 3), round(lesson_gap, 3))

## The dataset ladder
The same code runs from a hand-built D1 through real D4/D5 data. Each rung reports shape, classes, and a tiny sample so leakage, search, feature work, imbalance, and strategy choices stay inspectable.

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(clf_ladder(), 1):
    classes, counts = np.unique(y, return_counts=True)
    class_summary = ", ".join([f"{cls}:{cnt}" for cls, cnt in zip(classes, counts)])
    rows.append([f"D{idx}", name, str(X.shape), class_summary, np.array2string(X[:2, :min(3, X.shape[1])], precision=2)])
print_table(rows, ["rung", "name", "shape", "classes", "sample"])


## Run the same method across D1–D5
One metric is collected per rung so the summary curve is comparable.

In [ ]:

def leakage_experiment(X, y):
    x_tr, x_te, y_tr, y_te = safe_split(X, y)
    labels = np.unique(y)
    clean = Pipeline([
        ("scale", StandardScaler()),
        ("model", logistic_model())
    ])
    clean.fit(x_tr, y_tr)
    clean_loss = model_log_loss(clean, x_te, y_te, labels)

    rng = np.random.default_rng(43)
    leak_train = y_tr.reshape(-1, 1).astype(float)
    leak_test_wrong = y_te.reshape(-1, 1).astype(float)
    leak_test_realistic = rng.permutation(y_te).reshape(-1, 1).astype(float)
    x_tr_leak = np.column_stack([x_tr, leak_train])
    x_te_leak_wrong = np.column_stack([x_te, leak_test_wrong])
    x_te_leak_realistic = np.column_stack([x_te, leak_test_realistic])
    leaky = Pipeline([
        ("scale", StandardScaler()),
        ("model", logistic_model())
    ])
    leaky.fit(x_tr_leak, y_tr)
    leaky_loss = model_log_loss(leaky, x_te_leak_wrong, y_te, labels)
    deployed_loss = model_log_loss(leaky, x_te_leak_realistic, y_te, labels)
    gap = deployed_loss - leaky_loss
    return leaky_loss, clean_loss, deployed_loss, gap

rows = []
metrics = []
leakage_results = []
for idx, (name, X, y) in enumerate(clf_ladder(), 1):
    leaky_loss, clean_loss, deployed_loss, gap = leakage_experiment(X, y)
    leakage_results.append((name, leaky_loss, clean_loss, deployed_loss, gap))
    metrics.append(clean_loss)
    rows.append([f"D{idx}", f"{leaky_loss:.3f}", f"{clean_loss:.3f}", f"{deployed_loss:.3f}", f"{gap:.3f}"])
print_table(rows, ["rung", "leaky_val", "fixed_val", "deployed", "leak_gap"])


## Results visualization
The first five panels preview the ladder data; the last panel tracks the metric from D1 to D5.

In [ ]:

plot_summary([r[0] for r in leakage_results], metrics, "fixed validation loss vs rung", "log loss")


## Pitfall on the hardest rung
D5 is where a shortcut can look most convincing. The cell reproduces the wrong behavior and then applies the safer fix.

In [ ]:

name, leaky_loss, clean_loss, deployed_loss, gap = leakage_results[-1]
print(name)
print(f"wrong leaky validation loss: {leaky_loss:.4f}")
print(f"realistic deployed loss after leak disappears: {deployed_loss:.4f}")
print(f"fixed validation loss with leak removed: {clean_loss:.4f}")
assert leaky_loss < clean_loss
assert deployed_loss > leaky_loss



## Evaluate it + Practice
- Compare the reported metric with a no-skill baseline before trusting the method.
- Run a cheap sanity check: shuffle labels or remove the key idea and confirm performance drops.
- Ablate the technique in the table above and inspect whether D5 changes more than D1.
- Watch failure signals: unstable validation numbers, suspiciously perfect scores, or a gap that grows with complexity.

Practice prompts:
1. Change one hyperparameter or preprocessing choice and rerun the ladder.


2. Add a small amount of label noise to D3 and explain which metric moved first.

3. On D5, write one sentence explaining whether the method is reducing bias, variance, or evaluation error.